# Proyecto ML - Parte 1: Machine Learning Clásico

Este notebook construye la entrega final de la Parte 1 del proyecto. El objetivo es clasificar textos según su década de origen usando únicamente técnicas clásicas de aprendizaje automático con `scikit-learn`.

De acuerdo con el enunciado, no se usan redes neuronales, aprendizaje profundo, transformers, embeddings preentrenados ni modelos de lenguaje. Siguiendo el hint recibido para esta parte, el modelo final es una regresión logística dentro de un `Pipeline` con representación TF-IDF.


In [ ]:
import os

# Limita hilos para que el notebook sea estable en entornos locales o de evaluación.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1. Carga de datos

El conjunto `train.csv` contiene los textos etiquetados con `decade`. El conjunto `eval.csv` contiene los textos sin etiqueta y se usa únicamente para generar el archivo de respuesta de Kaggle.


In [ ]:
BASE_DIR = Path.cwd()
if not (BASE_DIR / "Data" / "train.csv").exists():
    BASE_DIR = BASE_DIR / "Proyecto-ML"

TRAIN_PATH = BASE_DIR / "Data" / "train.csv"
EVAL_PATH = BASE_DIR / "Data" / "eval.csv"
SUBMISSION_PATH = BASE_DIR / "submission_parte1.csv"
MODEL_PATH = BASE_DIR / "modelo_parte1.joblib"

train_df = pd.read_csv(TRAIN_PATH)
eval_df = pd.read_csv(EVAL_PATH)

print(f"train.csv: {train_df.shape}")
print(f"eval.csv: {eval_df.shape}")
print("Columnas train:", train_df.columns.tolist())
print("Columnas eval:", eval_df.columns.tolist())

display(train_df.head())
display(eval_df.head())


## 2. Exploración y limpieza mínima de datos

Antes de entrenar el modelo se revisan las columnas esperadas, tamaños, textos nulos o vacíos, etiquetas faltantes, duplicados y distribución de clases. La limpieza se limita a eliminar registros de entrenamiento sin texto, sin etiqueta o duplicados exactos. No se modifican acentos, grafías antiguas, signos, mayúsculas/minúsculas manualmente ni caracteres raros, porque esas variaciones pueden ser señales útiles para diferenciar décadas.


In [ ]:
expected_train_cols = {"text", "decade"}
expected_eval_cols = {"id", "text"}

assert expected_train_cols.issubset(train_df.columns), "train.csv no tiene las columnas esperadas: text, decade"
assert expected_eval_cols.issubset(eval_df.columns), "eval.csv no tiene las columnas esperadas: id, text"

quality_summary = pd.DataFrame({
    "conjunto": ["train", "eval"],
    "filas": [len(train_df), len(eval_df)],
    "textos_nulos": [train_df["text"].isna().sum(), eval_df["text"].isna().sum()],
    "textos_vacios": [train_df["text"].fillna("").str.strip().eq("").sum(), eval_df["text"].fillna("").str.strip().eq("").sum()],
    "etiquetas_o_ids_nulos": [train_df["decade"].isna().sum(), eval_df["id"].isna().sum()],
    "duplicados_exactos": [train_df.duplicated().sum(), eval_df.duplicated().sum()],
})

display(quality_summary)

# Limpieza mínima del conjunto de entrenamiento: solo se eliminan filas sin texto, sin etiqueta o duplicadas exactas.
train_model_df = train_df.dropna(subset=["text", "decade"]).copy()
train_model_df = train_model_df[train_model_df["text"].str.strip().ne("")].copy()
rows_before_dedup = len(train_model_df)
train_model_df = train_model_df.drop_duplicates().copy()
train_model_df["decade"] = train_model_df["decade"].astype(int)

# En evaluación no se eliminan filas porque Kaggle exige una predicción para cada id.
eval_model_df = eval_df.copy()
eval_model_df["text"] = eval_model_df["text"].fillna("")

print(f"Filas de train antes de limpieza: {len(train_df)}")
print(f"Filas de train después de eliminar nulos/vacíos: {rows_before_dedup}")
print(f"Duplicados exactos eliminados de train: {rows_before_dedup - len(train_model_df)}")
print(f"Filas de train después de limpieza mínima: {len(train_model_df)}")
print(f"Filas de eval conservadas: {len(eval_model_df)}")
print("Número de clases:", train_model_df["decade"].nunique())
print("Rango de décadas:", train_model_df["decade"].min(), "a", train_model_df["decade"].max())

display(train_model_df["decade"].value_counts().sort_index().to_frame("conteo").T)

text_lengths = train_model_df["text"].str.len()
length_summary = text_lengths.describe()[["min", "25%", "50%", "75%", "max", "mean"]].to_frame("longitud_texto").T
display(length_summary)


## 3. Preparación de datos

La partición local se usa solo para estimar el desempeño del pipeline final antes de entrenarlo con todos los datos etiquetados limpios. `eval.csv` no se usa en entrenamiento ni validación; únicamente se conserva para generar el archivo final con todos los `id`.


In [ ]:
X = train_model_df["text"]
y = train_model_df["decade"]
X_eval = eval_model_df["text"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Tamaño entrenamiento local:", X_train.shape[0])
print("Tamaño validación local:", X_val.shape[0])


## 4. Pipeline final

El pipeline final usa `TfidfVectorizer` con n-gramas de caracteres y `LogisticRegression`. Esta representación es adecuada para textos históricos porque captura patrones ortográficos y es robusta ante variaciones de escritura o ruido en el texto.


In [ ]:
final_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        sublinear_tf=True,
        min_df=2,
        max_df=0.95,
        max_features=200_000,
        lowercase=True,
    )),
    ("model", LogisticRegression(
        C=4.0,
        penalty="l2",
        solver="liblinear",
        dual=True,
        max_iter=2000,
        random_state=RANDOM_STATE,
    )),
])

final_pipeline



La configuración final se mantiene después de revisar las oportunidades de mejora propuestas. Se evaluaron variantes con regularización más fuerte (`C=0.1`, `0.5`, `1.0`), espacios de características más pequeños (`max_features=100_000`) o más restrictivos (`min_df=3`, `min_df=5`), `analyzer="char_wb"` y una combinación `FeatureUnion` de palabras y caracteres. En la partición local estratificada, ninguna de esas alternativas superó al pipeline `char` con n-gramas `(3, 5)`, `max_features=200_000`, `min_df=2` y `C=4.0`; por eso se conserva como modelo final.


## 5. Entrenamiento y evaluación local

Se entrena el pipeline final en la partición local de entrenamiento y se evalúa sobre la partición de validación.


In [ ]:
final_pipeline.fit(X_train, y_train)

y_pred_train = final_pipeline.predict(X_train)
y_pred_val = final_pipeline.predict(X_val)

metrics = pd.DataFrame({
    "conjunto": ["train", "validacion"],
    "accuracy": [
        accuracy_score(y_train, y_pred_train),
        accuracy_score(y_val, y_pred_val),
    ],
    "f1_macro": [
        f1_score(y_train, y_pred_train, average="macro"),
        f1_score(y_val, y_pred_val, average="macro"),
    ],
})

display(metrics)
print(classification_report(y_val, y_pred_val, zero_division=0))


El desempeño en validación se usa como estimación local. Para el envío final, el mismo pipeline se reentrena con todo `train.csv`, como corresponde para aprovechar todos los ejemplos etiquetados disponibles.


## 6. Entrenamiento final y archivo para Kaggle

Se reentrena el pipeline final con todos los datos etiquetados y se generan predicciones para `eval.csv`. El archivo de respuesta debe tener exactamente las columnas `id,answer`.


In [ ]:
final_pipeline.fit(X, y)

eval_predictions = final_pipeline.predict(X_eval)
submission_df = pd.DataFrame({
    "id": eval_model_df["id"],
    "answer": eval_predictions.astype(int),
})

assert list(submission_df.columns) == ["id", "answer"]
assert len(submission_df) == len(eval_df)
assert submission_df.isna().sum().sum() == 0
assert set(submission_df["answer"]).issubset(set(y.unique()))

submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"Archivo guardado en: {SUBMISSION_PATH}")
print(f"Filas: {len(submission_df)}")
display(submission_df.head())


## 7. Guardado del modelo final

El enunciado pide entregar un único modelo final de `scikit-learn`, correspondiente al mejor envío de Kaggle. Se guarda el pipeline completo para que incluya tanto la vectorización TF-IDF como el clasificador.


In [ ]:
joblib.dump(final_pipeline, MODEL_PATH)
loaded_pipeline = joblib.load(MODEL_PATH)

sample_predictions = loaded_pipeline.predict(X_eval.head(5))
print(f"Modelo guardado en: {MODEL_PATH}")
print("Predicciones de verificación:", sample_predictions)


## 8. Conclusión

El modelo final de la Parte 1 es un `Pipeline` de `scikit-learn` compuesto por `TfidfVectorizer` con n-gramas de caracteres y `LogisticRegression`. Este notebook genera los dos archivos necesarios para la entrega final:

- `submission_parte1.csv`: archivo para subir a Kaggle con columnas `id,answer`.
- `modelo_parte1.joblib`: pipeline final entrenado, para entregar en Bloque Neón junto con este notebook.
